# AI-Powered EV Digital Twin & Powertrain Blockset
### MATLAB Powertrain Blockset Fidelity in Python with Groq GenAI Optimization

This notebook implements:
1. **Vehicle Dynamics & Electric Powertrain Physics** (Aerodynamics, Rolling resistance, Gradient force, Acceleration ODE)
2. **Electric Motor Modeling & Efficiency Map** (Torque-Speed curve, Dual Y-axis Power characteristics, 2D/3D Efficiency Map)
3. **Real-World EV Master Dataset Integration** (175+ production electric vehicles)
4. **Groq LLM Autonomous Powertrain Optimizer** (Synthesizes 3 architectures, runs digital twin simulations, explains the winner)
5. **Performance Testing** (0-100 km/h acceleration sprint, hill climb gradeability, cruising energy consumption)

In [ ]:
# Install dependencies if running on Google Colab
# !pip install groq plotly pandas numpy scikit-learn

In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Import custom simulation modules
from ev_physics import VehicleParams, MotorParams, TransmissionParams, EVSimulationEngine
from ev_dataset import EVDatasetManager
from ev_ai_agent import EVAIAgent

print("✅ All EV Digital Twin simulation modules imported successfully!")

## 1. Load Master Dataset & Production EV Benchmarks

In [ ]:
dataset = EVDatasetManager("EV_master_dataset_all_vehicles.csv")
print(f"Indexed {len(dataset.df)} production electric vehicles across {dataset.df['Segment'].nunique()} segments.")
dataset.df.head(10)

## 2. Define Vehicle Virtual Model & Digital Twin Simulation

In [ ]:
# Configure our virtual EV digital twin
vehicle = VehicleParams(
    name="Virtual Model EV-Twin",
    mass_kg=1850.0,
    drag_coefficient_cd=0.25,
    frontal_area_m2=2.25,
    rolling_resistance_cr=0.010,
    battery_capacity_kwh=75.0
)

motor = MotorParams(
    motor_type="PMSM (Permanent Magnet Synchronous)",
    rated_power_kw=170.0,
    peak_torque_nm=340.0,
    max_speed_rpm=16000.0,
    peak_efficiency=0.965
)

transmission = TransmissionParams(
    gearbox_type="Single-Speed Reduction Gearbox",
    gear_ratio_1=8.65
)

engine = EVSimulationEngine(vehicle, motor, transmission)
print(f"Motor Base Speed: {engine.motor.base_speed_rpm:.0f} RPM")

## 3. MATLAB Powertrain Blockset Equivalent: Torque & Power Curves

In [ ]:
curve_data = engine.generate_torque_power_curve()

fig_tp = make_subplots(specs=[[{"secondary_y": True}]])
fig_tp.add_trace(go.Scatter(x=curve_data["speed_rpm"], y=curve_data["torque_nm"], name="Torque (Nm)", line=dict(color="#0071E3", width=3)), secondary_y=False)
fig_tp.add_trace(go.Scatter(x=curve_data["speed_rpm"], y=curve_data["power_kw"], name="Power (kW)", line=dict(color="#FF9500", width=3, dash="dash")), secondary_y=True)
fig_tp.add_vline(x=curve_data["base_speed_rpm"], line_dash="dot", line_color="gray", annotation_text="Base Speed")
fig_tp.update_layout(title="<b>Torque & Power Characteristics (Dual Y-Axis)</b>", xaxis_title="Speed (RPM)", height=450)
fig_tp.update_yaxes(title_text="Torque (Nm)", secondary_y=False)
fig_tp.update_yaxes(title_text="Power (kW)", secondary_y=True)
fig_tp.show()

## 4. MATLAB Powertrain Blockset Equivalent: 3D Motor Efficiency Map

In [ ]:
eff_map = engine.generate_efficiency_map(35, 35)
fig_3d = go.Figure(data=[go.Surface(z=eff_map["efficiency_grid"], x=eff_map["speed_rpm"], y=eff_map["torque_nm"], colorscale="Viridis")])
fig_3d.update_layout(title="<b>3D Motor Efficiency Map (η %)</b>", scene=dict(xaxis_title="Speed (RPM)", yaxis_title="Torque (Nm)", zaxis_title="Efficiency (%)"), height=500)
fig_3d.show()

## 5. Performance Simulations: Acceleration Sprint & Gradeability

In [ ]:
# 0-100 km/h acceleration sprint ODE simulation
sprint = engine.simulate_acceleration_sprint(100.0)
print(f"⏱️ 0-100 km/h Sprint Time: {sprint['zero_to_target_time_s']:.2f} seconds")
print(f"🚀 Peak Acceleration: {np.max(sprint['accel_ms2']):.2f} m/s^2 ({np.max(sprint['accel_ms2'])/9.81:.2f} g)")

# Incline Gradeability
grade_df = engine.simulate_gradeability()
grade_df

## 6. Groq GenAI Autonomous Multi-Configuration Optimization
The AI Agent changes design parameters, synthesizes 3 distinct configurations (Single-Speed Balanced, 2-Speed High-Efficiency, and High-Torque Performance), runs the digital twin simulations in parallel, and explains which configuration performs best.

In [ ]:
# Run Autonomous Multi-Config Optimization
ai_agent = EVAIAgent(api_key="your_groq_api_key_here")

similar_evs = dataset.find_similar_evs(vehicle.mass_kg, 180, 340, 25, k=4)
opt_results = ai_agent.run_multi_config_optimization(
    base_vehicle=vehicle,
    target_top_speed_kmh=180.0,
    target_gradeability_pct=25.0,
    benchmark_evs_context=similar_evs.to_string(index=False)
)

print("=== SIMULATION COMPARISON MATRIX ===")
print(pd.DataFrame(opt_results["configurations"]).to_string(index=False))

print("\n=== AI CHIEF ENGINEER'S VERDICT & RECOMMENDATION ===")
print(opt_results["ai_verdict"])